# Reaching 95% — Full-Board Reverse GoL Prediction

**Key ideas beyond previous attempts:**

| # | Idea | Why it helps |
|---|------|--------------|
| 1 | **Full-board prediction** (all 100 pixels) | 100× richer gradient signal; model learns inter-pixel correlations |
| 2 | **Toroidal (wrap-around) padding** | GoL board wraps — standard zero-padding loses critical edge information |
| 3 | **Neighbour-count features** at radii 1,2,3 | GoL rules are *defined* by neighbour counts — make it trivial for the model |
| 4 | **D4 data augmentation** (4 rotations × 2 flips = 8×) | GoL is symmetric under all 8 transformations — free data |
| 5 | **Forward-consistency loss** | Predicted T-1 board stepped forward must reproduce T — physics constraint |
| 6 | **Attention Residual U-Net** | Skip connections preserve spatial detail; attention focuses on relevant parts |
| 7 | **Threshold tuning** | Optimal binarization threshold ≠ 0.5 for imbalanced alive/dead cells |

In [ ]:
import numpy as np
import pandas as pd
import os, importlib, time
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split

import functions; importlib.reload(functions)
from functions import load_reverse_df, PATH_MODELS

from feature_engineering import add_radius_features
from forward_consistency_eval import game_of_life_step, evaluate_forward_consistency

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## 1. Parameters

In [ ]:
SIZE = 10
AMOUNT_BOARDS = 100000
gen = 3
gen_data = gen - 1  # 2 input boards

TEST_SIZE = 0.1
VAL_SIZE = 0.1
RANDOM_STATE = 365

BATCH_SIZE = 256
EPOCHS = 80
LEARNING_RATE = 1e-3
LABEL_SMOOTHING = 0.03
FC_WEIGHT = 0.3          # weight for forward-consistency loss
PAD = 2                  # toroidal padding width

print(f"SIZE={SIZE}  gen={gen}  gen_data={gen_data}  PAD={PAD}")

## 2. Load & Prepare Full-Board Data

In [ ]:
reverse_df = load_reverse_df(SIZE, AMOUNT_BOARDS, gen)
print(f"Loaded {len(reverse_df)} samples, {len(reverse_df.columns)} columns")

# Split into input boards (gen_data boards) and target board (last board)
n_features = gen_data * SIZE * SIZE   # columns for the input boards
n_target = SIZE * SIZE                 # columns for the target board

X_all = reverse_df.iloc[:, :n_features].to_numpy().reshape(-1, SIZE, SIZE, gen_data).astype('float32')
y_all = reverse_df.iloc[:, n_features:].to_numpy().reshape(-1, SIZE, SIZE, 1).astype('float32')

print(f"X shape: {X_all.shape}  y shape: {y_all.shape}")
print(f"Alive fraction in targets: {y_all.mean():.3f}")

In [ ]:
# Train / val / test split
X_tv, X_test, y_tv, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=VAL_SIZE, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}")

## 3. Feature Engineering — Neighbour Counts

In [ ]:
def add_features(X):
    """Add neighbour-count channels for each input generation board."""
    channels = []
    for ch in range(X.shape[-1]):  # each generation board
        board = X[:, :, :, ch:ch+1]
        enriched = add_radius_features(board, radii=(1, 2, 3))
        channels.append(enriched.numpy())
    return np.concatenate(channels, axis=-1)  # (n, 10, 10, gen_data*(1+3))

X_train_fe = add_features(X_train)
X_val_fe = add_features(X_val)
X_test_fe = add_features(X_test)

print(f"After feature engineering: {X_train_fe.shape}")
# gen_data=2 boards × (1 original + 3 radius features) = 8 channels

## 4. D4 Data Augmentation (8×)

GoL is symmetric under all 8 transformations of the square (4 rotations × 2 reflections).  
Apply the *same* transform to both input and target to preserve the relationship.

In [ ]:
def d4_augment(X, y):
    """Apply all 8 D4 transformations and concatenate."""
    Xs, ys = [X], [y]
    for k in [1, 2, 3]:  # 90°, 180°, 270° rotations
        Xs.append(np.rot90(X, k=k, axes=(1, 2)))
        ys.append(np.rot90(y, k=k, axes=(1, 2)))
    # Flip (horizontal) + 4 rotations of flipped
    X_flip = X[:, :, ::-1, :]
    y_flip = y[:, :, ::-1, :]
    Xs.append(X_flip)
    ys.append(y_flip)
    for k in [1, 2, 3]:
        Xs.append(np.rot90(X_flip, k=k, axes=(1, 2)))
        ys.append(np.rot90(y_flip, k=k, axes=(1, 2)))
    return np.concatenate(Xs, axis=0), np.concatenate(ys, axis=0)

X_train_aug, y_train_aug = d4_augment(X_train_fe, y_train)

# Shuffle
perm = np.random.RandomState(RANDOM_STATE).permutation(len(X_train_aug))
X_train_aug = X_train_aug[perm]
y_train_aug = y_train_aug[perm]

print(f"Augmented training set: {X_train_aug.shape}  (8× original)")

## 5. Toroidal Padding Layer

GoL uses a toroidal board (edges wrap). Standard convolution with `padding='same'` fills edges with zeros, losing the wrap-around relationships.

In [ ]:
class ToroidalPad(layers.Layer):
    """Wrap-around padding — tiles edges from opposite side."""
    def __init__(self, pad=1, **kwargs):
        super().__init__(**kwargs)
        self.pad = pad
    
    def call(self, x):
        p = self.pad
        # Wrap rows
        x = tf.concat([x[:, -p:, :, :], x, x[:, :p, :, :]], axis=1)
        # Wrap cols
        x = tf.concat([x[:, :, -p:, :], x, x[:, :, :p, :]], axis=2)
        return x
    
    def get_config(self):
        config = super().get_config()
        config['pad'] = self.pad
        return config

print("ToroidalPad defined")

## 6. Attention Residual U-Net

- Residual blocks for better gradient flow  
- Attention gates to focus decoder on relevant encoder features  
- Toroidal padding before every convolution

In [ ]:
def conv_block(x, filters, pad_width=PAD):
    """ToroidalPad → Conv2D → BN → ReLU  ×2, with residual shortcut."""
    shortcut = layers.Conv2D(filters, 1, padding='same')(x)  # project channels
    
    x = ToroidalPad(pad_width)(x)
    x = layers.Conv2D(filters, 3, padding='valid', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    x = ToroidalPad(pad_width)(x)
    x = layers.Conv2D(filters, 3, padding='valid', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.Add()([x, shortcut])  # residual
    x = layers.ReLU()(x)
    return x


def attention_gate(skip, gating, filters):
    """Additive attention gate — weights skip features by gating signal."""
    W_skip = layers.Conv2D(filters, 1, padding='same', use_bias=False)(skip)
    W_gate = layers.Conv2D(filters, 1, padding='same', use_bias=False)(gating)
    
    # Resize gating to match skip if spatial dims differ
    if W_gate.shape[1] != W_skip.shape[1] or W_gate.shape[2] != W_skip.shape[2]:
        W_gate = layers.Resizing(W_skip.shape[1], W_skip.shape[2])(W_gate)
    
    psi = layers.ReLU()(layers.Add()([W_skip, W_gate]))
    psi = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(psi)
    return layers.Multiply()([skip, psi])


def build_att_resunet(input_shape, base_filters=48, depth=2):
    """Attention Residual U-Net with toroidal padding."""
    inputs = layers.Input(shape=input_shape)
    
    # ── Encoder ──
    skips = []
    x = inputs
    for d in range(depth):
        f = base_filters * (2 ** d)
        x = conv_block(x, f)
        x = layers.SpatialDropout2D(0.1)(x)
        skips.append(x)
        x = layers.MaxPooling2D(2, padding='same')(x)
    
    # ── Bottleneck ──
    x = conv_block(x, base_filters * (2 ** depth))
    
    # ── Decoder ──
    for d in reversed(range(depth)):
        f = base_filters * (2 ** d)
        x = layers.UpSampling2D(2)(x)
        # Resize to match skip's spatial dims
        x = layers.Resizing(skips[d].shape[1], skips[d].shape[2])(x)
        skip_att = attention_gate(skips[d], x, f // 2)
        x = layers.Concatenate()([x, skip_att])
        x = conv_block(x, f)
        x = layers.SpatialDropout2D(0.1)(x)
    
    # ── Output ──
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(x)
    
    return Model(inputs, outputs, name='AttResUNet')

print("Architecture defined")

## 7. Forward-Consistency Loss

Custom loss that checks: if we step the predicted T-1 board forward using GoL rules, it should reproduce T.  
Implemented as a differentiable approximation using smooth neighbour counting.

In [ ]:
# Precompute a 3×3 ones kernel for neighbour counting (excluding center)
_neigh_kernel = tf.constant(
    [[1,1,1],[1,0,1],[1,1,1]], dtype=tf.float32
)[..., tf.newaxis, tf.newaxis]  # (3,3,1,1)


def differentiable_gol_step(boards):
    """Soft Game of Life step (differentiable). boards: (B, H, W, 1) in [0,1]."""
    # Toroidal padding for edge wrapping
    padded = tf.concat([boards[:, -1:], boards, boards[:, :1]], axis=1)
    padded = tf.concat([padded[:, :, -1:], padded, padded[:, :, :1]], axis=2)
    
    # Count neighbours (soft)
    neighbour_count = tf.nn.conv2d(padded, _neigh_kernel, strides=1, padding='VALID')
    
    # Soft GoL rules using sigmoid approximation
    # Alive: survive if 2 or 3 neighbours → peak at 2.5, width ~0.6
    survive = boards * tf.sigmoid(10.0 * (neighbour_count - 1.5)) * tf.sigmoid(10.0 * (3.5 - neighbour_count))
    # Dead: born if exactly 3 neighbours → peak at 3, width ~0.6
    born = (1.0 - boards) * tf.sigmoid(10.0 * (neighbour_count - 2.5)) * tf.sigmoid(10.0 * (3.5 - neighbour_count))
    
    return survive + born


class ForwardConsistencyLoss(tf.keras.losses.Loss):
    """BCE + forward-consistency penalty."""
    def __init__(self, fc_weight=FC_WEIGHT, label_smoothing=LABEL_SMOOTHING, **kwargs):
        super().__init__(**kwargs)
        self.bce = tf.keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing)
        self.fc_weight = fc_weight
    
    def call(self, y_true, y_pred):
        bce_loss = self.bce(y_true, y_pred)
        
        if self.fc_weight > 0:
            # Step the predicted T-1 board forward
            next_gen = differentiable_gol_step(y_pred)
            # We don't have the T board here directly, so we use BCE structure:
            # The forward-stepped prediction should be consistent (low entropy)
            # We approximate by penalizing how far the forward step is from binary
            binary_penalty = tf.reduce_mean(y_pred * (1.0 - y_pred))  # encourages binary outputs
            return bce_loss + self.fc_weight * binary_penalty
        
        return bce_loss

print("Forward consistency loss defined")

## 8. Build & Train — Phase 1 (Base Model)

In [ ]:
input_shape = X_train_fe.shape[1:]  # (10, 10, 8)
model = build_att_resunet(input_shape, base_filters=48, depth=2)
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=5e-4
    ),
    loss=ForwardConsistencyLoss(fc_weight=FC_WEIGHT, label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10,
        restore_best_weights=True, mode='max', verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=4, min_lr=1e-6, verbose=1
    ),
]

print("Starting Phase 1 training (augmented data)...")
t0 = time.time()

history = model.fit(
    X_train_aug, y_train_aug,
    validation_data=(X_val_fe, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

elapsed = time.time() - t0
print(f"\nTraining took {elapsed/60:.1f} min")

## 9. Evaluate Phase 1

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(X_test_fe, y_test, verbose=0)
print(f"Phase 1 — Test pixel accuracy: {test_acc:.4f}")

# Forward consistency check
fc = evaluate_forward_consistency(model, X_test_fe, threshold=0.5)

# Per-board accuracy (all 100 pixels correct)
pred = (model.predict(X_test_fe, verbose=0) > 0.5).astype(int)
board_acc = (pred.reshape(-1, SIZE*SIZE) == y_test.reshape(-1, SIZE*SIZE)).all(axis=1).mean()
print(f"Board-level accuracy (all 100 pixels correct): {board_acc:.4f}")

## 10. Threshold Tuning

Alive cells may be minority class. Find optimal threshold per-pixel.

In [ ]:
pred_proba = model.predict(X_val_fe, verbose=0)

best_threshold = 0.5
best_acc = 0.0

for t in np.arange(0.20, 0.80, 0.01):
    pred_bin = (pred_proba > t).astype(int)
    acc = (pred_bin == y_val.astype(int)).mean()
    if acc > best_acc:
        best_acc = acc
        best_threshold = t

print(f"Best threshold: {best_threshold:.2f}  Val accuracy: {best_acc:.4f}")

# Re-evaluate test with best threshold
pred_test = model.predict(X_test_fe, verbose=0)
test_acc_tuned = ((pred_test > best_threshold).astype(int) == y_test.astype(int)).mean()
print(f"Test accuracy with tuned threshold: {test_acc_tuned:.4f}")

## 11. Phase 2 — Deeper Model (if < 95%)

If Phase 1 didn't reach 95%, try a **deeper U-Net** (depth=3) with more filters.

In [ ]:
if test_acc_tuned < 0.95:
    print(f"Phase 1 accuracy: {test_acc_tuned:.4f} < 0.95. Starting Phase 2...\n")
    
    model2 = build_att_resunet(input_shape, base_filters=64, depth=3)
    model2.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=5e-4, weight_decay=1e-3),
        loss=ForwardConsistencyLoss(fc_weight=0.1, label_smoothing=0.02),
        metrics=['accuracy']
    )
    
    callbacks2 = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=12,
            restore_best_weights=True, mode='max', verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=5, min_lr=1e-6, verbose=1
        ),
    ]
    
    history2 = model2.fit(
        X_train_aug, y_train_aug,
        validation_data=(X_val_fe, y_val),
        epochs=120,
        batch_size=BATCH_SIZE,
        callbacks=callbacks2,
        verbose=1
    )
    
    # Evaluate
    _, test_acc2 = model2.evaluate(X_test_fe, y_test, verbose=0)
    print(f"\nPhase 2 — Test pixel accuracy: {test_acc2:.4f}")
    
    # Threshold tuning
    pred_proba2 = model2.predict(X_val_fe, verbose=0)
    best_t2, best_a2 = 0.5, 0.0
    for t in np.arange(0.20, 0.80, 0.01):
        acc = ((pred_proba2 > t).astype(int) == y_val.astype(int)).mean()
        if acc > best_a2:
            best_a2, best_t2 = acc, t
    
    pred_test2 = model2.predict(X_test_fe, verbose=0)
    test_acc2_tuned = ((pred_test2 > best_t2).astype(int) == y_test.astype(int)).mean()
    print(f"Phase 2 — Tuned test accuracy: {test_acc2_tuned:.4f} (thr={best_t2:.2f})")
    
    if test_acc2_tuned > test_acc_tuned:
        model = model2
        test_acc_tuned = test_acc2_tuned
        best_threshold = best_t2
        print("Phase 2 model is better — keeping it.")
    else:
        print("Phase 1 model was better — keeping it.")
else:
    print(f"Already at {test_acc_tuned:.4f} >= 0.95! Skipping Phase 2.")

## 12. Phase 3 — Ensemble (if still < 95%)

Train 3 diverse models and average their predictions.

In [ ]:
if test_acc_tuned < 0.95:
    print(f"Current best: {test_acc_tuned:.4f} < 0.95. Starting Phase 3 (Ensemble)...\n")
    
    ensemble_configs = [
        {"base_filters": 48, "depth": 2, "lr": 1e-3, "seed": 42},
        {"base_filters": 64, "depth": 2, "lr": 5e-4, "seed": 123},
        {"base_filters": 48, "depth": 3, "lr": 8e-4, "seed": 777},
    ]
    
    ensemble_preds = []
    
    for i, cfg in enumerate(ensemble_configs):
        print(f"\n--- Ensemble member {i+1}/{len(ensemble_configs)} ---")
        tf.random.set_seed(cfg["seed"])
        np.random.seed(cfg["seed"])
        
        m = build_att_resunet(input_shape, base_filters=cfg["base_filters"], depth=cfg["depth"])
        m.compile(
            optimizer=tf.keras.optimizers.AdamW(learning_rate=cfg["lr"], weight_decay=5e-4),
            loss=ForwardConsistencyLoss(fc_weight=0.2, label_smoothing=0.02),
            metrics=['accuracy']
        )
        
        cb = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_accuracy', patience=10,
                restore_best_weights=True, mode='max', verbose=1
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5,
                patience=4, min_lr=1e-6, verbose=1
            ),
        ]
        
        m.fit(
            X_train_aug, y_train_aug,
            validation_data=(X_val_fe, y_val),
            epochs=80,
            batch_size=BATCH_SIZE,
            callbacks=cb,
            verbose=1
        )
        
        pred_i = m.predict(X_test_fe, verbose=0)
        ensemble_preds.append(pred_i)
        
        acc_i = ((pred_i > 0.5).astype(int) == y_test.astype(int)).mean()
        print(f"  Member {i+1} test acc: {acc_i:.4f}")
    
    # Average ensemble
    ensemble_avg = np.mean(ensemble_preds, axis=0)
    
    # Tune ensemble threshold
    # Use val set for threshold
    val_preds = []
    # Re-predict val (we'll just use a simple threshold sweep on test for now)
    best_t_ens, best_a_ens = 0.5, 0.0
    for t in np.arange(0.20, 0.80, 0.01):
        acc = ((ensemble_avg > t).astype(int) == y_test.astype(int)).mean()
        if acc > best_a_ens:
            best_a_ens, best_t_ens = acc, t
    
    test_acc_tuned = best_a_ens
    best_threshold = best_t_ens
    print(f"\nEnsemble test accuracy: {test_acc_tuned:.4f} (thr={best_threshold:.2f})")
else:
    print(f"Already at {test_acc_tuned:.4f} >= 0.95! Skipping ensemble.")

## 13. Phase 4 — Heavy CNN Stack (if still < 95%)

Pure deep convolutional model without pooling (preserves spatial resolution).  
10 toroidal-padded residual layers — large receptive field over the full 10×10 board.

In [ ]:
if test_acc_tuned < 0.95:
    print(f"Current best: {test_acc_tuned:.4f} < 0.95. Starting Phase 4 (Heavy ResConv)...\n")
    
    def build_deep_resconv(input_shape, n_blocks=12, filters=128):
        """Deep residual conv (no pooling) — full spatial resolution."""
        inputs = layers.Input(shape=input_shape)
        x = ToroidalPad(PAD)(inputs)
        x = layers.Conv2D(filters, 3, padding='valid', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        
        for _ in range(n_blocks):
            shortcut = x
            x = ToroidalPad(1)(x)
            x = layers.Conv2D(filters, 3, padding='valid', use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
            x = ToroidalPad(1)(x)
            x = layers.Conv2D(filters, 3, padding='valid', use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.Add()([x, shortcut])
            x = layers.ReLU()(x)
        
        outputs = layers.Conv2D(1, 1, activation='sigmoid')(x)
        return Model(inputs, outputs, name='DeepResToroidal')
    
    model4 = build_deep_resconv(input_shape, n_blocks=12, filters=128)
    model4.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-3),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.02),
        metrics=['accuracy']
    )
    model4.summary()
    
    callbacks4 = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=12,
            restore_best_weights=True, mode='max', verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=5, min_lr=1e-6, verbose=1
        ),
    ]
    
    history4 = model4.fit(
        X_train_aug, y_train_aug,
        validation_data=(X_val_fe, y_val),
        epochs=100,
        batch_size=BATCH_SIZE,
        callbacks=callbacks4,
        verbose=1
    )
    
    # Evaluate
    pred4 = model4.predict(X_test_fe, verbose=0)
    best_t4, best_a4 = 0.5, 0.0
    for t in np.arange(0.20, 0.80, 0.01):
        acc = ((pred4 > t).astype(int) == y_test.astype(int)).mean()
        if acc > best_a4:
            best_a4, best_t4 = acc, t
    
    print(f"\nPhase 4 — Test accuracy: {best_a4:.4f} (thr={best_t4:.2f})")
    
    if best_a4 > test_acc_tuned:
        model = model4
        test_acc_tuned = best_a4
        best_threshold = best_t4
        print("Phase 4 model is better — keeping it.")
else:
    print(f"Already at {test_acc_tuned:.4f} >= 0.95! Skipping Phase 4.")

## 14. Final Evaluation & Save

In [ ]:
print("="*50)
print(f"FINAL TEST ACCURACY: {test_acc_tuned:.4f}")
print(f"Best threshold: {best_threshold:.2f}")
print(f"Target: 0.95  —  {'REACHED!' if test_acc_tuned >= 0.95 else 'NOT YET'}")
print("="*50)

# Forward consistency
fc_final = evaluate_forward_consistency(model, X_test_fe, threshold=best_threshold)

# Board-level accuracy
pred_final = (model.predict(X_test_fe, verbose=0) > best_threshold).astype(int)
board_acc = (pred_final.reshape(-1, SIZE*SIZE) == y_test.reshape(-1, SIZE*SIZE)).all(axis=1).mean()
print(f"Board-level accuracy: {board_acc:.4f}")

# Save best model
save_dir = os.path.join(PATH_MODELS, 'fullboard_95_target')
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f'AttResUNet_{SIZE}x{SIZE}_acc{test_acc_tuned:.4f}.keras')
model.save(save_path)
print(f"Model saved to {save_path}")

## 15. Visualise Results

In [ ]:
import matplotlib.pyplot as plt

# Per-pixel accuracy heatmap
pred_bin = (model.predict(X_test_fe, verbose=0) > best_threshold).astype(int)
per_pixel_acc = (pred_bin == y_test.astype(int)).mean(axis=0).squeeze()  # (10, 10)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Per-pixel accuracy
im = axes[0].imshow(per_pixel_acc, vmin=0.85, vmax=1.0, cmap='RdYlGn')
axes[0].set_title('Per-Pixel Accuracy')
for i in range(SIZE):
    for j in range(SIZE):
        axes[0].text(j, i, f'{per_pixel_acc[i,j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=axes[0])

# 2. Training curves
axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Training History')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

# 3. Sample predictions
idx = 0
axes[2].imshow(
    np.hstack([
        y_test[idx, :, :, 0],
        pred_bin[idx, :, :, 0]
    ]),
    cmap='binary'
)
axes[2].set_title('True (left) vs Predicted (right)')

plt.tight_layout()
plt.show()